## Load libraries

In [5]:
import numpy as np
import pandas as pd

from pywebio.input import *
from pywebio.output import *
from pywebio import start_server
from pywebio.exceptions import SessionClosedException

import pickle
import warnings
import argparse

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split

## Load the model

In [6]:
# Load the model pipeline from the file
with open('nlp_pipeline.pkl', 'rb') as f:
    loaded_pipe = pickle.load(f)

c:\Users\Natasha Maina\anaconda3\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator CountVectorizer from version 1.3.2 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Natasha Maina\anaconda3\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.3.2 when using version 1.5.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Natasha Maina\anaconda3\Lib\site-packages\sklearn\base.py:376: InconsistentVersionWarning: Trying to unpickle estimator LogisticRegression from version 1.3.2 when using version 1.5

## Function to preprocess text

In [7]:
import string
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

# function to remove punctuation and stopwords
def text_process(mess):
    """
    Takes in a string of text, then performs the following:
    1. Remove all punctuation
    2. Remove all stopwords
    3. Returns a list of the cleaned text
    """
    STOPWORDS = stopwords.words('english') #.words('english') + ['u', 'ü', 'ur', '4', '2', 'im', 'dont', 'doin', 'ure']
    # Check characters to see if they are in punctuation
    nopunc = [char for char in mess if char not in string.punctuation]

    # Join the characters again to form the string.
    nopunc = ''.join(nopunc)
    
    # Now just remove any stopwords
    return ' '.join([word for word in nopunc.split() if word.lower() not in STOPWORDS])

[nltk_data] Downloading package stopwords to C:\Users\Natasha
[nltk_data]     Maina\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Pywebio App

In [ ]:
# Function to predict the insurance charges
def prediction(prediction_df):
    pred_out = loaded_pipe.predict(prediction_df)
    final_result = pred_out[0]

    return final_result


sms = pd.read_csv('spam.csv', encoding='latin-1')  #
sms.dropna(how="any", inplace=True, axis=1)
sms.columns = ['label', 'message']
sms['label_num'] = sms.label.map({'ham': 0, 'spam': 1})
sms['clean_msg'] = sms.message.apply(text_process)

# Split the data
X = sms.clean_msg
y = sms.label_num
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=1)

# Build and train the pipeline
pipe = Pipeline([
    ('bow', CountVectorizer()),  
    ('tfid', TfidfTransformer()),  
    ('model', MultinomialNB())  
])

# Train the model
pipe.fit(X_train, y_train)


# Function to get user input and display the prediction results
def main():
    put_markdown(
        '''
        # Application for predicting if a clients response is a Spam or Genuine message
        '''
        , lstrip=True
    )

    model_inputs = input_group(
        "Enter client's message below:",
        [
            textarea("Enter your text here", name='message'),
        ]
    )

    # save user input in a dataframe and prepare it for prediction
    prediction_df = pd.DataFrame(data = [[model_inputs[i] for i in ['message']]], 
                           columns = ['message'])
    
    # use text_process function to remove punctuations and stopwords
    prediction_df['cleaned_message'] = prediction_df.message.apply(text_process)

    user_input_message = prediction_df.cleaned_message
    message_input = prediction(user_input_message)
    put_markdown("### This response has been marked as: {} ".format(message_input))


    # Display the prediction results in a table
    prediction_prob = pipe.predict_proba(prediction_df.cleaned_message)[0]
    put_table([
        ['Label', 'Probability'],
        ['Ham', f"{prediction_prob[0]:.2f}"],
        ['Spam', f"{prediction_prob[1]:.2f}"]
    ])


# function to Start the PyWebIO web application
if __name__ == "__main__":
    try:
        main()
    except SessionClosedException:
        print("The session was closed unexpectedly")

The session was closed unexpectedly


In [5]:
# #inp = ["YOU JUST WON A MILLION DOLLARS! TO CLAIM PRIZE, CALL 123 NOW! HURRY UP!"]
# inp = ["hi Jon, how are you doing today?"]

In [10]:
#Save model to pkl file
# Save the model to a .pkl file - ADD THIS IN A NEW CELL
with open('spam_classifier.pkl', 'wb') as f:
    pickle.dump(pipe, f)
print("Model successfully saved to 'spam_classifier.pkl'")

Model successfully saved to 'spam_classifier.pkl'
